In [21]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import os


In [114]:
# Load data
prices_data = pd.read_csv('../../data/raw/features/prices_2019_15102025.csv')
ml_targets = pd.read_csv("../../data/processed/sl_ml_targets_2025-10-06.csv")
hh_info = pd.read_csv("../../data/processed/hh_info.csv")
adm2_average = pd.read_csv("../../data/processed/adm2_average.csv")

In [86]:
# load the data
def load_data(filepath):
    try: 
        return pd.read_csv(filepath)
    except Exception as e: 
        print(f"Error loading {filepath}: {e}")
        return None

# pre process data 
def preprocess_prices(df):
    """Filter and pivot price data for analysis."""
    filtered = df[df['Data Type'] != "Forecast"]
    grouped = (
        filtered.groupby(['Admin 2', 'Commodity'])
        .agg({'Price': 'median'})
        .reset_index()
    )
    pivoted = grouped.pivot(index='Admin 2', columns='Commodity', values='Price')
    return pivoted

Functions to create plots for EA

In [ ]:
def plot_heatmap(pivot_data, title="Prices of Commodities per Admin 2", cmap="Blues"):
    plt.figure(figsize=(10, 6))
    sns.heatmap(pivot_data, cmap=cmap)
    plt.title(title)
    plt.xlabel("Commodity")
    plt.ylabel("Admin 2")
    plt.tight_layout()
    plt.show()


def plot_histograms(pivot_data, commodities=None, country_label='LKA'):
    """Plot histograms for multiple commodities."""
    if commodities is None:
        commodities = pivot_data.columns  # Use all available commodities

    for commodity in commodities:
        if commodity in pivot_data.columns:
            plt.figure(figsize=(10, 6))
            sns.histplot(data=pivot_data, x=commodity, bins=20, kde=True)
            plt.xlabel(f"{commodity} price ({country_label})")
            plt.ylabel("Count")
            plt.title(f"Distribution of {commodity} Prices")
            plt.tight_layout()
            plt.show()
        else:
            print(f"Commodity '{commodity}' not found in data.")



def plot_data(df, commodities=None, country_label='LKA'):
    """Main analysis function: preprocess and plot."""
    if df is not None:
        pivot_data = preprocess_prices(df)
        plot_heatmap(pivot_data, title=f"Prices of Commodities per Admin 2 ({country_label})")
        plot_histograms(pivot_data, commodities=commodities, country_label=country_label)
        return pivot_data
    else:
        print("No data provided.")
        return None



Apply the analysis in one function

In [ ]:

def produce_plots(filepaths):
    """Processes a list of file paths and returns analysis results."""
    results = {}
    for path in filepaths:
        try:
            df = load_data(path)  # Assumes you have a load_data() function
            analysis = plot_data(df)
            filename = os.path.basename(path)
            results[filename] = analysis
            print(f"Processed: {filename}")
        except Exception as e:
            print(f"Error processing {path}: {e}")
            results[os.path.basename(path)] = None
    return results


print the results

In [ ]:

files = ['../../data/raw/features/prices_2024_03112025.csv', '../../data/raw/features/prices_2019_15102025.csv']  # Replace with your actual file paths
results = produce_plots(files)

for filename, result in results.items():
    # print(f"\nAnalysis for {filename}:\n")
    print(result)


## Prices vs intake for 2019 data

In [126]:

def clean_prices_data(prices_data):
    prices_data = preprocess_prices(prices_data)
    prices_data.columns = [str(col) for col in prices_data.columns]
    return prices_data.reset_index()


In [127]:

def merge_district_codes(prices_data, adm2_average):
    districts = [
        {"adm2": 81, "Admin 2": "Badulla"}, {"adm2": 11, "Admin 2": "Colombo"},
        {"adm2": 12, "Admin 2": "Gampaha"}, {"adm2": 41, "Admin 2": "Jaffna"},
        {"adm2": 13, "Admin 2": "Kalutara"}, {"adm2": 21, "Admin 2": "Kandy"},
        {"adm2": 92, "Admin 2": "Kegalle"}, {"adm2": 61, "Admin 2": "Kurunegala"},
        {"adm2": 43, "Admin 2": "Mannar"}, {"adm2": 22, "Admin 2": "Matale"},
        {"adm2": 82, "Admin 2": "Moneragala"}, {"adm2": 23, "Admin 2": "Nuwara Eliya"},
        {"adm2": 72, "Admin 2": "Polonnaruwa"}, {"adm2": 91, "Admin 2": "Ratnapura"},
        {"adm2": 53, "Admin 2": "Trincomalee"}, {"adm2": 44, "Admin 2": "Vavuniya"}
    ]
    df_districts = pd.DataFrame(districts)
    prices_data=prices_data.merge(df_districts, on='Admin 2')

    adm2_inad = (
        adm2_average[['adm2', 'energy_kcal_q50'] + [col for col in adm2_average.columns if col.endswith('_inad')]]
        .merge(prices_data, on='adm2', how='left')
    )


    return adm2_inad
  

In [128]:

def add_province_names(df):
    province_map = {
        1: "Western", 2: "Central", 3: "Southern", 4: "Northern", 5: "Eastern",
        6: "North Western", 7: "North Central", 8: "Uva", 9: "Sabaragamuwa"
    }
    df['province'] = (df['adm2'] // 10).round().map(province_map)
    return df


In [129]:

def define_variable_groups(df):
    mn_col_names = list(df.columns[2:8])
    price_col_names = list(df.columns[11:15])
    return mn_col_names, price_col_names


In [130]:
def plot_scatter_with_regression(df, x_var, y_var, output_dir="outputs/plots/prices"):
    print(f"Plotting {y_var} vs {x_var}")
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=df, x=x_var, y=y_var, hue='province', s=50)
    sns.regplot(data=df, x=x_var, y=y_var, scatter=False, color='black')

    # Clean data
    X = sm.add_constant(df[x_var])
    y = df[y_var]
    data = pd.concat([X, y], axis=1).replace([np.inf, -np.inf], np.nan).dropna()
    X_clean = data.iloc[:, :-1]
    y_clean = data.iloc[:, -1]

    # Fit model
    model = sm.OLS(y_clean, X_clean).fit()
    r_squared = model.rsquared
    intercept, slope = model.params
    pearson_r = slope / abs(slope) * r_squared**0.5

    # Annotate
    eq_label = f"$y = {intercept:.2f} + {slope:.2f}x$\n$R^2 = {r_squared:.2f}$\n$Pearson = {pearson_r:.2f}$"
    plt.text(0.05, 0.95, eq_label, transform=plt.gca().transAxes,
             verticalalignment='top', horizontalalignment='left', fontsize=10)

    plt.title(f"Scatterplot of {y_var} vs {x_var}")
    plt.tight_layout()
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(f"{output_dir}/{y_var}_{x_var}.png")
    plt.close()

In [140]:

def analyze_relationships(adm2_average, prices_data):
    prices_data = clean_prices_data(prices_data)
    prices_data = merge_district_codes(prices_data, adm2_average)
    adm2_inad = add_province_names(prices_data)
    mn_col_names, price_col_names = define_variable_groups(adm2_inad)

    for y_var in mn_col_names:
        for x_var in price_col_names:
            plot_scatter_with_regression(adm2_inad, x_var, y_var)


In [141]:

prices_data = pd.read_csv('../../data/raw/features/prices_2019_15102025.csv')
analyze_relationships(adm2_average, prices_data)


Plotting vita_inad vs Rice (long grain)
Plotting vita_inad vs Rice (white)


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\regression\linear_model.py:1781: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


Plotting vita_inad vs Sugar
Plotting vita_inad vs Wheat flour
Plotting zn_inad vs Rice (long grain)
Plotting zn_inad vs Rice (white)


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\regression\linear_model.py:1781: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


Plotting zn_inad vs Sugar
Plotting zn_inad vs Wheat flour
Plotting folate_inad vs Rice (long grain)
Plotting folate_inad vs Rice (white)


...

Plotting thia_inad vs Sugar
Plotting thia_inad vs Wheat flour
Plotting vitb12_inad vs Rice (long grain)
Plotting vitb12_inad vs Rice (white)


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\regression\linear_model.py:1781: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


Plotting vitb12_inad vs Sugar
Plotting vitb12_inad vs Wheat flour
Plotting fe_inad vs Rice (long grain)
Plotting fe_inad vs Rice (white)


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\regression\linear_model.py:1781: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


Plotting fe_inad vs Sugar
Plotting fe_inad vs Wheat flour
